<div style="background-color: black;">
<hr style="border: 3px solid skyblue;">
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 20px; font-family: TimesNewRoman; color: skyblue">
    TIME SERIES DATA PROCESSING
<br>
    CROSS BORDER FLOWS
</div>
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color: skyblue">
    Main Formatting Notebook
    <br>
    from PYPSA to DISPA-SET
</div>
<div style="text-align: justify; margin-left: 0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color: skyblue">
This script processes raw PyPSA simulation data to generate cross-border flow time series for every country modeled in the Dispa-SET Unleashed project.
<br>
Refer to the explanation cells for a step-by-step guide from raw data processing to final output.
</div>
    <hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    1. Notebook Set Up
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Importing needed libraries.
<hr style="border: 1px solid skyblue;">
</div>
</div>

In [281]:
import os
import csv
from datetime import datetime
import requests
import pandas as pd
from shutil import move
import numpy as np
import shutil
from bs4 import BeautifulSoup
import re
import io
import plotly.graph_objects as go
from typing import List, Dict, Tuple, Optional, Tuple
import re
from IPython.display import HTML
from difflib import get_close_matches
from collections import defaultdict
import warnings
from pathlib import Path
import glob
from itertools import permutations
from itertools import combinations

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Auxiliar Code
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cell has the purpose to create the correponding folders with the name of all the EU countries available in the ENTSOE data base.
    <br>
    Uncomment it to use it just if needed.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [150]:
'''
# List of countries with their acronyms in parentheses
countries = [
    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",
    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",
    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",
    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",
    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",
    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",
    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",
    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",
    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",
    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slovenia   (SI)",
    "Spain         (ES)", "Sweden         (SE)", "Switzerland      (CH)", "Turkey     (TR)",
    "Ukraine       (UA)", "United Kingdom (UK)"
]

# Set the path where you want to create the folders
base_path = '/home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/HydroData/ScaledInflows'

# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)

# Loop through the list of countries
for country_string in countries:
    # Use a regular expression to find the acronym inside the parentheses
    match = re.search(r'\((.*?)\)', country_string)

    # If a match is found, extract the acronym
    if match:
        acronym = match.group(1).strip()  # Use strip() to remove any extra whitespace
        folder_path = os.path.join(base_path, acronym)

        # Check if the folder already exists to avoid errors
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")
    else:
        print(f"Could not extract acronym from: {country_string}")

print("\nAll folders created successfully!")
'''

<>:1: SyntaxWarning: invalid escape sequence '\('
<>:1: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_2243005/2889290989.py:1: SyntaxWarning: invalid escape sequence '\('
  '''


'\n# List of countries with their acronyms in parentheses\ncountries = [\n    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",\n    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",\n    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",\n    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",\n    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",\n    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",\n    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",\n    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",\n    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",\n    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slove

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    2. Dispa-SET_Unleash Folder Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Determinning dynamically the zone_folder_path based on the location of the "Dispa-SET_Unleash" folder relative to the current working directory. 
<br>
    If the "Dispa-SET_Unleash" folder is copied to a different machine or location, the dispaSET_unleash_folder_path variable will automatically adjust accordingly.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [151]:
# 1. ------------------------------------------------------------------------------  Get the current working directory 
current_directory = os.getcwd()

# 2. --------------------------------------------------------  Navigate to the parent directory of "Dispa-SET_Unleash" 
dispaSET_unleash_parent_directory = os.path.dirname(current_directory)

# 3. ------------------------------------------------------------------ Get the path to the "Dispa-SET_Unleash" folder 
dispaSET_unleash_folder_path = os.path.dirname(dispaSET_unleash_parent_directory)

# 4. ------------------------------------------------------------- Construct the dispaSET_unleash_folder_name variable 
dispaSET_unleash_folder_name = os.path.basename(dispaSET_unleash_folder_path)

# 5. ------------------------------------------------------------------ ----------------------------------------- Done 
print("dispaSET_unleash_folder_name:", dispaSET_unleash_folder_name)
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)

dispaSET_unleash_folder_name: Dispa-SET_Unleash
dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.1. PyPSA Source Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
There are two sccenarios as source of PyPSA power plants raw data.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
Reference_Scenario
<li>
Suficiency_Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The scenario variable must be selected before proceeding to the next processing steps.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [152]:
# 1. ----------------------------------------------------------------------------------------- Set the needed scenatio 
pypsa_scenario = "Reference_Scenario"
#pypsa_scenario = "Suficiency_Scenario"

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("PyPSA selected Scenario:", pypsa_scenario)

PyPSA selected Scenario: Reference_Scenario


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.2. Dispa-SET Time Step
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The time series data must be resampled to a predetermined time step.
<br>
The UNLEASH project utilizes three levels of granularity:
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
One hour (1h) 
<li>
Thirty minutes (30min)
<li>
Fifteen minutes (15min)
</li>
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [153]:
# 1. ---------------------------------------------------------------- Set the time step to which data is formating to: 
data_target_time_step = '1h'
# data_target_time_step = '15min'
# data_target_time_step = '30min'

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Selected time step:", data_target_time_step)

Selected time step: 1h


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.3. Secondary Folder Paths
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The many subfolders within the Unleash directory require their location paths to be defined for correct access during processing.
<br>
All of these are dependent on the chosen scenario.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [154]:
# 1. ---------------------- Get the path to the "Dispa-SET_Unleash_Cross_Border_Flows" folder of the data used as base 
additional_path_1 = os.path.join("/Database/CrossBorderFlows", data_target_time_step)
cross_border_flows_base_data_folder_path = dispaSET_unleash_folder_path + additional_path_1

# 2. ---------------- Construct the "Dispa-SET_Unleash_Cross_Border_Flows" name variable of the data used as reference 
cross_border_flows_base_data_folder_name = os.path.basename(cross_border_flows_base_data_folder_path)

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("cross_border_flows_base_data_folder_name:", cross_border_flows_base_data_folder_name)
print("cross_border_flows_base_data_folder_path:", cross_border_flows_base_data_folder_path)
# ===========================================================================================================================================

# 1. ------------------------- Get the path to the "Dispa-SET_Unleash_Cross_Border_Flows" folder of the PyPSA raw data 
additional_path_2 = os.path.join("RawData_PyPSA", pypsa_scenario, "CrossBorderFlows")
cross_border_flows_pypsa_raw_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_2

# 2. ------------------- Construct the Dispa-SET_Unleash_Cross_Border_Flows_folder_name variable of the PyPSA raw data 
cross_border_flows_pypsa_raw_data_folder_name = os.path.basename(cross_border_flows_pypsa_raw_data_folder_path)

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("cross_border_flows_pypsa_raw_data_folder_name:", cross_border_flows_pypsa_raw_data_folder_name)
print("cross_border_flows_folder_path:", cross_border_flows_pypsa_raw_data_folder_path)
# ===========================================================================================================================================

# 1. -------------------- Get the path to the "Dispa-SET_Unleash_Cross_Border_Flows" folder of the PyPSA formated data 
additional_path_3 = os.path.join("Database_PyPSA", pypsa_scenario, "CrossBorderFlows", data_target_time_step)
cross_border_flows_pypsa_formated_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_3

# 2. ------------ Construct the "Dispa-SET_Unleash_Cross_Border_Flows" folder name variable of the PyPSA formated data 
cross_border_flows_pypsa_formated_data_folder_name = os.path.basename(cross_border_flows_pypsa_formated_data_folder_path)

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("cross_border_flows_pypsa_formated_data_folder_name:", cross_border_flows_pypsa_formated_data_folder_name)
print("cross_border_flows_pypsa_formated_data_folder_path:", cross_border_flows_pypsa_formated_data_folder_path)

cross_border_flows_base_data_folder_name: 1h
cross_border_flows_base_data_folder_path: /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/1h
cross_border_flows_pypsa_raw_data_folder_name: CrossBorderFlows
cross_border_flows_folder_path: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/CrossBorderFlows
cross_border_flows_pypsa_formated_data_folder_name: 1h
cross_border_flows_pypsa_formated_data_folder_path: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/CrossBorderFlows/1h


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    3. Zone(s) Creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Entering the zone name or names (comment those ones that are not available data) where all data related to the corresponding zone are going to be storage.
<br>
For European country names use the ISO 3166-1 standars i.e. AT, BE, BG, CH.... etc. to give the zone_name.
</div>
<hr style="border: 1px solid skyblue;">

In [155]:
# 1. -------------------------------------------------------------------- Set the list of folder names to be addressed 
zone_names = [
                #"AL",
                #"AM",
                #"AT",
                #"AZ",
                #"BY",
                "BE",
                #"BA",
                #"BG",
                #"HR",
                #"CY",
                #"CZ",
                #"DK",
                #"EE",
                #"FI",
                "FR",
                #"GE",
                "DE",
                #"EL",
                #"HU",
                #"IS",
                #"IE",
                #"IT",
                #"XK",
                #"LV",
                #"LT",
                #"LU",
                #"MT",
                #"MD",
                #"ME",
                "NL",
                #"MK",
                #"NO",
                #"PL",
                #"PT",
                #"RO",
                #"RU",
                #"RS",
                #"SK",
                #"SI",
                #"ES",
                #"SE",
                #"CH",
                #"TR",
                #"UA",
                "UK"
             ]

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Selected zones:", zone_names)

Selected zones: ['BE', 'FR', 'DE', 'NL', 'UK']


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The following dictionary specifies the possible alternative names—or synonyms/aliases—used in international nomenclature for the EU countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [156]:
# 1. --------------------------------------------------------- Set the list of the alternative acronyms per EU country 
zone_names_equivalences_dict = {

"AL"  :  {"Acronym": ["  "       ] ,   "name": ["Albania   "       ]},  
"AM"  :  {"Acronym": ["  "       ] ,   "name": ["Armenia   "       ]},
"AT"  :  {"Acronym": ["  "       ] ,   "name": ["Austria   "       ]},
"AZ"  :  {"Acronym": ["  "       ] ,   "name": ["Azerbaijan"       ]},
"BY"  :  {"Acronym": ["  "       ] ,   "name": ["Belarus"          ]},
"BE"  :  {"Acronym": ["  "       ] ,   "name": ["Belgium"          ]},
"BA"  :  {"Acronym": ["  "       ] ,   "name": ["Bosnia and Herz." ]},
"BG"  :  {"Acronym": ["  "       ] ,   "name": ["Bulgaria"         ]},
"HR"  :  {"Acronym": ["  "       ] ,   "name": ["Croatia"          ]},
"CY"  :  {"Acronym": ["  "       ] ,   "name": ["Cyprus"           ]},
"CZ"  :  {"Acronym": ["  "       ] ,   "name": ["Czech Republic"   ]},
"DK"  :  {"Acronym": ["  "       ] ,   "name": ["Denmark"          ]},
"EE"  :  {"Acronym": ["  "       ] ,   "name": ["Estonia"          ]},
"FI"  :  {"Acronym": ["  "       ] ,   "name": ["Finland"          ]},
"FR"  :  {"Acronym": ["  "       ] ,   "name": ["France"           ]},
"GE"  :  {"Acronym": ["  "       ] ,   "name": ["Georgia"          ]}, 
"DE"  :  {"Acronym": ["  "       ] ,   "name": ["Germany"          ]},
"EL"  :  {"Acronym": ["GR"       ] ,   "name": ["Greece"           ]},
"HU"  :  {"Acronym": ["  "       ] ,   "name": ["Hungary"          ]},
"IS"  :  {"Acronym": ["  "       ] ,   "name": ["Iceland"          ]},
"IE"  :  {"Acronym": ["  "       ] ,   "name": ["Ireland"          ]},
"IT"  :  {"Acronym": ["  "       ] ,   "name": ["Italy"            ]},
"XK"  :  {"Acronym": ["  "       ] ,   "name": ["Kosovo"           ]},
"LV"  :  {"Acronym": ["  "       ] ,   "name": ["Latvia"           ]},
"LT"  :  {"Acronym": ["  "       ] ,   "name": ["Lithuania"        ]},
"LU"  :  {"Acronym": ["  "       ] ,   "name": ["Luxembourg"       ]},
"MT"  :  {"Acronym": ["  "       ] ,   "name": ["Malta"            ]},
"MD"  :  {"Acronym": ["  "       ] ,   "name": ["Moldova"          ]},
"ME"  :  {"Acronym": ["  "       ] ,   "name": ["Montenegro"       ]},
"NL"  :  {"Acronym": ["  "       ] ,   "name": ["Netherlands"      ]},
"MK"  :  {"Acronym": ["  "       ] ,   "name": ["North Macedonia"  ]},
"NO"  :  {"Acronym": ["  "       ] ,   "name": ["Norway"           ]},
"PL"  :  {"Acronym": ["  "       ] ,   "name": ["Poland"           ]},
"PT"  :  {"Acronym": ["  "       ] ,   "name": ["Portugal"         ]},
"RO"  :  {"Acronym": ["  "       ] ,   "name": ["Romania"          ]},
"RU"  :  {"Acronym": ["  "       ] ,   "name": ["Russia"           ]},
"RS"  :  {"Acronym": ["  "       ] ,   "name": ["Serbia"           ]},
"SK"  :  {"Acronym": ["  "       ] ,   "name": ["Slovakia"         ]},
"SI"  :  {"Acronym": ["  "       ] ,   "name": ["Slovenia"         ]},
"ES"  :  {"Acronym": ["  "       ] ,   "name": ["Spain"            ]},
"SE"  :  {"Acronym": ["  "       ] ,   "name": ["Sweden"           ]},
"CH"  :  {"Acronym": ["  "       ] ,   "name": ["Switzerland"      ]},
"TR"  :  {"Acronym": ["  "       ] ,   "name": ["Turkey"           ]},
"UA"  :  {"Acronym": ["  "       ] ,   "name": ["Ukraine"          ]},
"UK"  :  {"Acronym": ["GB"       ] ,   "name": ["United Kingdom"   ]},
       
}

# 2. ------------------------------------------------ Create a new dictionary with only the keys present in zone_names
selected_zone_names_equivalences_dict = {
    key: zone_names_equivalences_dict[key]
    for key in zone_names
    if key in zone_names_equivalences_dict
}

# 3. ------------------------------------------------------------------------------------------------------------ Done 
for key, value in selected_zone_names_equivalences_dict.items():
    if any(acronym.strip() for acronym in value["Acronym"]):
        print(f"Key: {key};    Acronym: {value['Acronym']};    Name: {value['name']}\n")

Key: UK;    Acronym: ['GB'];    Name: ['United Kingdom']



<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    4. Data Reference Year 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Setting the variable on the target year which formatting data is wanted to
</div>
<hr style="border: 1px solid skyblue;">

In [157]:
# 1. --------------------------------------------------------------------- Set the year to which data is formating to: 
data_target_year = '2030'

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Selected Year:", data_target_year)

Selected Year: 2030


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [158]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Cross Border Flows Base data folder:           {cross_border_flows_base_data_folder_name}\n")
print (f"Path to the Cross Border Flows Base data folder:           {cross_border_flows_base_data_folder_path}\n")
print (f"Name of the Cross Border Flows_Pypsa Raw data 1 folder:    {cross_border_flows_pypsa_raw_data_folder_name}\n")
print (f"Path to the Cross Border Flows_Pypsa Raw data 1 folder:    {cross_border_flows_pypsa_raw_data_folder_path}\n")
print (f"Name of the Cross Border Flows_Pypsa Formated data folder: {cross_border_flows_pypsa_formated_data_folder_name}\n")
print (f"Path to the Cross Border Flows_Pypsa Formated data folder: {cross_border_flows_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):           {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Target time step:                                          {data_target_time_step}\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Cross Border Flows Base data folder:           1h

Path to the Cross Border Flows Base data folder:           /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/1h

Name of the Cross Border Flows_Pypsa Raw data 1 folder:    CrossBorderFlows

Path to the Cross Border Flows_Pypsa Raw data 1 folder:    /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/CrossBorderFlows

Name of the Cross Border Flows_Pypsa Formated data folder: 1h

Path to the Cross Border Flows_Pypsa Formated data folder: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/CrossBorderFlows/1h

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Name of the zone_names_equivalences_dictionary):           [['  '], ['  '], ['  '], ['  '], ['GB']]

Target year:                              

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
4. Cross Border FLows Data Frame
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating the data frame with all the corresponding headers according the Dispa-SET nomenclature.
</div>
<div style="text-align: justify; margin-left: 4.5em; font-weight: bold; font-size: 15px; font-family: TimesNewRoman; color:skyblue">
Sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
The Dispa-SET documentation.
            <br>
            <a href="https://www.dispaset.eu/en/latest/data.html#:~:text=Interconnections%EF%83%81" style="color:skyblue">https://www.dispaset.eu/en/latest/data.html#:~:text=Interconnections%EF%83%81</a>
        </li>
</div>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
4.1. Empty Zone Data Frames
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating empty dataframes—column headers only—for the selected zones, corresponding to the target year.
<br>
Loading the csv bade file.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [159]:
# 1. ------------------------------------------------------------------------------------ Convert data year to integer 
data_target_year = int(data_target_year)  

# 2. ----------------------------------------------------------------------------- Find the most appropriate csv files
csv_files = [
    f for f in os.listdir(cross_border_flows_base_data_folder_path)
    if f.lower().endswith(".csv")
]

# 3. --------------------------------------------------------------------- Keep only filenames that are exactly a year
available_years = {
    int(os.path.splitext(f)[0]): f
    for f in csv_files
    if os.path.splitext(f)[0].isdigit()
}

if not available_years:
    raise ValueError("No CSV files named with a year found")

# 4. ------------------------------------------------------------------------------- Select exact year or closest year
if data_target_year in available_years:
    selected_year = data_target_year
else:
    selected_year = min(
        available_years.keys(),
        key=lambda y: abs(y - data_target_year)
    )

selected_csv_path = os.path.join(
    cross_border_flows_base_data_folder_path,
    available_years[selected_year]
)

# 5. ----------------------------------------------------------------------------------- Load csv file into data frame
cross_border_flows_df = pd.read_csv(selected_csv_path)

# 6. ------------------------------------------------------------------------------------------------------------ Done
print(f"CSV year used: {selected_year}")
print(cross_border_flows_df.columns.tolist())
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_df'.")
cross_border_flows_df

CSV year used: 2024
['index', 'BG -> RoW', 'RoW -> BG', 'EE -> RoW', 'RoW -> EE', 'EL -> RoW', 'RoW -> EL', 'FI -> RoW', 'RoW -> FI', 'HR -> RoW', 'RoW -> HR', 'HU -> RoW', 'RoW -> HU', 'IT -> RoW', 'RoW -> IT', 'LT -> RoW', 'RoW -> LT', 'LV -> RoW', 'RoW -> LV', 'PL -> RoW', 'RoW -> PL', 'RO -> RoW', 'RoW -> RO', 'SK -> RoW', 'RoW -> SK']
✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_df'.


,index,BG -> RoW,RoW -> BG,EE -> RoW,RoW -> EE,EL -> RoW,RoW -> EL,FI -> RoW,RoW -> FI,HR -> RoW,...,LT -> RoW,RoW -> LT,LV -> RoW,RoW -> LV,PL -> RoW,RoW -> PL,RO -> RoW,RoW -> RO,SK -> RoW,RoW -> SK
0,2024-01-01 00:00:00+00:00,827.0,0.0,0.0,109.0,0.0,1129.0,0.0,0.0,534.0,...,0.0,68.0,178.0,0.0,0.0,0.0,159.0,367.0,340.0,0.0
1,2024-01-01 01:00:00+00:00,852.0,0.0,0.0,83.0,0.0,1170.0,0.0,0.0,466.0,...,0.0,72.0,178.0,0.0,0.0,0.0,223.0,283.0,247.0,0.0
2,2024-01-01 02:00:00+00:00,823.0,0.0,0.0,255.0,0.0,1158.0,0.0,0.0,334.0,...,43.0,0.0,220.0,0.0,0.0,0.0,302.0,204.0,197.0,0.0
3,2024-01-01 03:00:00+00:00,808.0,0.0,0.0,174.0,0.0,1178.0,0.0,0.0,184.0,...,0.0,20.0,192.0,0.0,0.0,0.0,425.0,159.0,138.0,0.0
4,2024-01-01 04:00:00+00:00,675.0,0.0,0.0,156.0,1.0,1174.0,0.0,0.0,77.0,...,0.0,47.0,190.0,0.0,0.0,0.0,512.0,154.0,140.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,2024-12-31 19:00:00+00:00,42.0,191.0,0.0,51.0,279.0,58.0,0.0,0.0,1048.0,...,95.0,0.0,71.0,0.0,425.0,0.0,273.0,821.0,965.0,0.0
8780,2024-12-31 20:00:00+00:00,131.0,145.0,0.0,81.0,387.0,79.0,0.0,0.0,923.0,...,107.0,0.0,59.0,0.0,397.0,0.0,136.0,624.0,935.0,0.0
8781,2024-12-31 21:00:00+00:00,227.0,120.0,0.0,163.0,412.0,144.0,0.0,0.0,1129.0,...,77.0,0.0,50.0,0.0,445.0,0.0,238.0,638.0,965.0,0.0
8782,2024-12-31 22:00:00+00:00,343.0,46.0,0.0,213.0,278.0,212.0,0.0,0.0,1326.0,...,133.0,0.0,25.0,0.0,366.0,0.0,279.0,699.0,976.0,0.0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Identifying the interconnections between the selected zones/countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [160]:
# 1. ------------------------------------------------------------------------- Assuming zone_names is the original list
zone_names_with_row = zone_names + ['RoW']

# 2. ------------------------------------------------ Build all ordered zone combinations (A -> B), now including 'RoW'
zone_combinations = [f"{a} -> {b}" for a, b in permutations(zone_names_with_row, 2)]
zone_combinations_set = set(zone_combinations)

# 3. ----------------------------------------------------------- Identify columns that look like a combination (X -> Y)
combination_like_cols = [
    col for col in cross_border_flows_df.columns
    if isinstance(col, str) and " -> " in col
]

# 4. ------------------------------------------------------ Among the combination-like columns, keep only exact matches
matching_cols = [col for col in combination_like_cols if col in zone_combinations_set]

# 5. ---------------------------------------------------------------- Remove combination-like columns that do NOT match
cols_to_drop = set(combination_like_cols) - set(matching_cols)
cross_border_flows_df = cross_border_flows_df.drop(columns=cols_to_drop)

# 6. --------------------------------------------------------------------- Create missing combinations as empty columns
missing_combinations = zone_combinations_set - set(matching_cols)
for col in missing_combinations:
    cross_border_flows_df[col] = np.nan

# 7. --------------------------------------- Sort columns if desired: keep original order + new combinations at the end
cross_border_flows_df = cross_border_flows_df[list(cross_border_flows_df.columns)]

# 8. ------------------------------------------------------------------------------------------------------------- Done
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_df'.")
cross_border_flows_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_df'.


,index,RoW -> BE,DE -> FR,DE -> RoW,NL -> DE,RoW -> NL,BE -> FR,RoW -> FR,UK -> BE,FR -> BE,...,DE -> BE,DE -> NL,RoW -> DE,UK -> FR,UK -> RoW,BE -> DE,BE -> NL,FR -> RoW,FR -> UK,NL -> UK
0,2024-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-01 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-01 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-01 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-01-01 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,2024-12-31 19:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8780,2024-12-31 20:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8781,2024-12-31 21:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8782,2024-12-31 22:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
4.2. Time Step Correction
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Verifying and correcting first column timestamps in the dataframe to match target year and selected time step.
<hr style="border: 1px solid skyblue;">
</div>

In [161]:
# 1. ---------------------------------------------- Define a mapping for time step strings to pandas frequency strings
time_step_map = {
    "1h"   : "H",
    "15min": "15T",
    "30min": "30T"
}

# 2. -------------------------------------------------------- Get the pandas frequency string for the target time step
freq = time_step_map[data_target_time_step]

# Assume `df` is your single DataFrame
if cross_border_flows_df.empty:
    print("DataFrame is empty. Skipping alignment.")
else:
    
# 2.1 -------------------------------------------------------------------------------------- Identify the first column
    first_col = cross_border_flows_df.columns[0]

# 2.2 ------------------------------------------------------------- Convert column to datetime with UTC if not already
    cross_border_flows_df[first_col] = pd.to_datetime(cross_border_flows_df[first_col], utc=True, errors='coerce')

# 2.3 --------------------------------------------------------------------------- Remove any rows that failed to parse
    cross_border_flows_df = cross_border_flows_df.dropna(subset=[first_col])

# 2.4 --------------------------------------------------------- Create a date range for the correct year and time step
    start_time = pd.Timestamp(f"{data_target_year}-01-01 00:00:00", tz="UTC")
    end_time = pd.Timestamp(f"{data_target_year}-12-31 23:59:59", tz="UTC")

    correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)

# 2.5 --------------------------------------------------------- Replace the first column with the corrected date range
    if len(correct_index) >= len(cross_border_flows_df):
        cross_border_flows_df[first_col] = correct_index[:len(cross_border_flows_df)]
    else:
        
# 2.5.1 ----------------------------------------- If df has more rows than the date range, extend with repeated values
        repeats = (len(cross_border_flows_df) // len(correct_index)) + 1
        cross_border_flows_df[first_col] = pd.Series(list(correct_index) * repeats)[:len(cross_border_flows_df)]

# 2.6 ----------------------------------------------------------------------------------------------------------- Done
    print(f"Updated DataFrame: first column aligned with {data_target_year} and {data_target_time_step}")

Updated DataFrame: first column aligned with 2030 and 1h


/tmp/ipykernel_2243005/1556487799.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [162]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Cross Border Flows Base data folder:           {cross_border_flows_base_data_folder_name}\n")
print (f"Path to the Cross Border Flows Base data folder:           {cross_border_flows_base_data_folder_path}\n")
print (f"Name of the Cross Border Flows_Pypsa Raw data 1 folder:    {cross_border_flows_pypsa_raw_data_folder_name}\n")
print (f"Path to the Cross Border Flows_Pypsa Raw data 1 folder:    {cross_border_flows_pypsa_raw_data_folder_path}\n")
print (f"Name of the Cross Border Flows_Pypsa Formated data folder: {cross_border_flows_pypsa_formated_data_folder_name}\n")
print (f"Path to the Cross Border Flows_Pypsa Formated data folder: {cross_border_flows_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):           {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Target time step:                                          {data_target_time_step}\n")
print (f"Name of the Cross Border Flows DataFrame:                  {list(cross_border_flows_df.keys())}\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Cross Border Flows Base data folder:           1h

Path to the Cross Border Flows Base data folder:           /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/1h

Name of the Cross Border Flows_Pypsa Raw data 1 folder:    CrossBorderFlows

Path to the Cross Border Flows_Pypsa Raw data 1 folder:    /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Reference_Scenario/CrossBorderFlows

Name of the Cross Border Flows_Pypsa Formated data folder: 1h

Path to the Cross Border Flows_Pypsa Formated data folder: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/CrossBorderFlows/1h

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Name of the zone_names_equivalences_dictionary):           [['  '], ['  '], ['  '], ['  '], ['GB']]

Target year:                              

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: 'Times New Roman', serif; color: skyblue;">
5. PyPSA vs Dispaset Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
All those technologies from PyPSA which can be represented in Dispa-SET have to be identified.
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.1. PyPSA Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
The next chart represents graphically how all the Energy sector inside PyPSA is structured.<br>
This is used to get the equivalent diagram for Dispaset.
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_multisector_figure_1.png" 
       alt="PyPSA Multisector Flow Diagram" 
       style="max-width:35%; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: <a href="https://pypsa-eur.readthedocs.io/en/latest/" target="_blank" style="color: skyblue; text-decoration: underline;">PyPSA-Eur Documentation</a>
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.2. Equivalent Dispa-SET Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
A correlation has been established between the PyPSA parameters and their corresponding Dispa-SET equivalents:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
Due to feature limitations, not all elements from PyPSA can be represented in Dispa-SET.
<br>
However, for those compatible technologies, the following chart graphically illustrates how they are connected within the Dispa-SET environment logic:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_Filtered_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.3. Technologies, Demmands & Interconnection Lines Nomenclature - PyPSA vs Dispaset 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The PyPSA technologies and demmands nomenclature and their correlation with their homologous from Dispa-SET are described as follows:
</div>
<table style="width: 95%; margin-left: auto; margin-right: auto; border-collapse: collapse; font-family: TimesNewRoman; font-size: 12px; color: skyblue;">
  <thead>
    <tr style="background-color: #1E1E1E; color: skyblue; border-bottom: 1px solid skyblue;">
      <th style="width: 8%; padding: 8px; text-align: left; border: 1px solid #444;">PyPSA Element</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Technology</th>
      <th style="width: 10%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Classification</th>
      <th style="width: 34%; padding: 8px; text-align: left; border: 1px solid #444;">Description</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Relation</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Dispa-SET Element</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Element Type</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">May Modeled?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DC</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents the DC (HVDC) transmission network for electricity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">NTC</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">OCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Open-Cycle Gas Turbine producing electricity from gas.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined-Cycle Gas Turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">EV charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Interface between the grid and electric vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">V2G</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Vehicle-to-Grid---allows EVs to discharge electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to stored energy in batteries (charging link)</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">BioSNG</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces synthetic natural gas (bio-methane)---fuel synthesis process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DAC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct Air Capture---captures CO<sub>2</sub> for storage or utilization</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Fischer-Tropsch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + CO<sub>2</sub> to liquid hydrocarbons; fuel synthesis</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Electrolysis</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity into hydrogen cross-sector conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Fuel Cell</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts hydrogen back to electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports hydrogen between regions or sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline retrofitted</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Existing pipelines adapted for H<sub>2</sub> transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Generates electricity or heat from hydrogen---boundary technology</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Haber-Bosch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + N<sub>2</sub> into ammonia---chemical/fertilizer industry</td>
      <td style="padding: 8px; border: 1px solid #444;">PX2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Steam Methane Reforming---gas to hydrogen conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR with Carbon Capture---industrial hydrogen with CC</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Sabatier</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> + CO<sub>2</sub> → CH<sub>4</sub>---synthetic methane production.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture machinery oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil used in agricultural machinery---transport/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">ammonia cracker</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts ammonia back into hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored electricity from batteries</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity distribution grid</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents distribution-level power flow---low voltage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">nuclear</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear-to-electricity conversion within the power system.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Upgrades raw biogas into pipeline-quality methane</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biogas upgrading with carbon capture---industrial fuel conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biomass to liquid</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass into liquid fuels---synthetic fuel process.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents captured Tons of CO<sub>2</sub> / hour transported or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Use of coal as industrial feedstock/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Supplies natural gas to industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial gas use with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports natural gas---energy carrier infrastructure</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline new</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Expansion of natural gas transport capacity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">kerosene for aviation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Aviation fuel consumption---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil use for land transport---transport fuel consumption</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">methanolisation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Synthesizes methanol (CO<sub>2</sub> + H<sub>2</sub> → CH<sub>3</sub>OH)</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides naphtha feedstock to industrial processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial process CO<sub>2</sub> emissions. Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial CO<sub>2</sub> emissions with capture — Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for rural homes</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass to heat---residential fuel use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---non-electric final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Uses electricity directly for heating---part of demand side</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to thermal energy in storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat — part of residential heating loop</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized air-source heat pump for urban buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating devices---boundary heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">link & residential rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heat using stable ground temperature</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
     <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges heat from thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Service sector rural building heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides space or process heat for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity-to-heat for service buildings---heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating in rural service buildings.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating for service buildings---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for service‐sector thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to buildings---part of the heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Local electric heat production for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass‐to‐heat conversion---end-use heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct electric heating for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts power to stored heat</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Releases stored thermal energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Methanol use in maritime transport---fuel demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption for ships---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass fuel use for industrial heat/processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same as above but with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass transport</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents biomass logistics between regions/sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">FlowXmaximum & FlowXminimum</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized district heat pump---heat sector interfac.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined heat + power supplying district heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Gas CHP with carbon capture---district heating system</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized gas heating for urban networks</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric boiler for district heating---end-use conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass combined heat + power---heat boundary process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same with carbon capture---boundary sector</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat in district storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to the district network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fossil fuel-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas–fired power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal variant used for power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
        <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Powerx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">onwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Onshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-ac</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with AC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-dc</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with DC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Utility-scale photovoltaic generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar rooftop</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed PV connected to power grid</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">ror</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Run-of-river hydro power plant</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel input for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces heat for households (not electricity)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized solar heating for buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Solar thermal for service-sector heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban service-sector solar heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized solar thermal for district heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">load</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents total shredding energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Load Shedding</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">hydro</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional hydro reservoir</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">PHS</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Pumped Hydro Storage, a grid-scale electricity storage technology</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">battery</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electrical energy storage — directly coupled with the grid.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
            <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel stock for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen storage---chemical energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">NH<sub>3</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Ammonia storage---chemical/fertilizer or fuel vector.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomethane stock for heating or industry</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Captured CO<sub>2</sub> pool---used in synthesis or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Permanent CO<sub>2</sub> storage---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>      
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> stored</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Intermediate or final CO<sub>2</sub> reservoir---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal stock for industrial/fuel processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas (CH<sub>4</sub>) stock — cross-sector energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fuel storage for thermal use — outside grid operations</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Liquid fuel stock — used in transport or synthesis chains</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil (synthetic hydrocarbons) stock for transport/industrial fuels</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass stock for heating/industrial use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for rural households — heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1评审44;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed heat storage in urban residences</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat storage for rural service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for urban service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating storage — boundary heat network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Final oil demand in the industrial sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary DH demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating demand for residential and service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized residential space/water heating demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in lighting, irrigation, machinery)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat energy required in agricultural processes (drying, greenhouses)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption in agricultural machinery and vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">aviation oil demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Jet fuel (kerosene) demand for aviation transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand for rail network</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Traction electricity used by rail and metro transport systems</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand of residential and tertairy</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in households and service-sector buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity use for machinery, processes, and electrified production</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">methane</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas or synthetic methane Industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">hydrogen for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand for industrial refining, ammonia, steelmaking</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport EV</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption by electric vehicles in road transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport hydrogen demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen fuel demand for road transport (fuel-cell vehicles)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">low-temperature heat for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">(below $\sim 200^{\circ}$C), typically supplied by boilers or heat pumps</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">Non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Naphtha used as a chemical feedstock e.g., plastics, petrochemicals</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">oil to transport demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil demand for conventional land gasoline and diesel vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand, maritime transport fuel-cell/ combustion ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional marine oil fuel demand (HFO, MGO) for ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass demand in industrial processes for heat or material use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
  </tbody>
</table>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 0.5px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.3. Dispa-SET vs PyPSA Interconnection Lines Equivalences Dictionary
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
To facilitate data harmonization, a dictionary containing the correlations between the raw sources files and the Dispa-SET and PyPSA nomenclature equivalences for interconnection lines is developed, leveraging the specifications detailed in the preceding table.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [212]:
# 1. ------------------------------------------------------- Dictionary mapping Dispa-SET acronyms to PyPSA tech names 
tech_equivalences_dict = {
    
                            # -----------------------
                            # Interconection lines
                            # -----------------------
                            "Tech_AC"  :  {"transmission_lines": ["AC"  ] ,   "capacities": ["AC Transmission lines"  ],   "RM_Factor": ["0.898"  ] } ,
    
                            "Tech_DC"  :  {"transmission_lines": ["DC"  ] ,   "capacities": ["DC Transmission lines"  ],   "RM_Factor": ["1.000"  ] } ,
       
                          }

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print(tech_equivalences_dict)

{'Tech_AC': {'transmission_lines': ['AC'], 'capacities': ['AC Transmission lines'], 'RM_Factor': ['0.898']}, 'Tech_DC': {'transmission_lines': ['DC'], 'capacities': ['DC Transmission lines'], 'RM_Factor': ['1.000']}}


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
        6. PyPSA to Dispa-SET Country Interconnections Flows Data Formatting
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Formatting the raw cross-border flow DataFrame.
    </div>
    <hr style="border: 1px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
6.1. Interconnection Data Sources
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Uploading the raw data into data frames for the formating process.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [304]:
# 1. ------------------------------------------------------------------------------------------- Define inner variables
folder_path = cross_border_flows_pypsa_raw_data_folder_path

# 2. -------------------------------------------------------- Identify files that contain the target year in their name
files_to_read = [
    f for f in os.listdir(folder_path) 
    if f.endswith('.csv') and str(data_target_year) in f
]

#  2.1. ----------------------------------------------------------------- Sort files to ensure consistent merging order
files_to_read.sort()

# 3. ------------------------------------------------------------------------------------ Process and combine the files
cross_border_flows_raw_data_df = pd.DataFrame()

for i, file_name in enumerate(files_to_read):
    file_full_path = os.path.join(folder_path, file_name)
    temp_df = pd.read_csv(file_full_path)
    
    if cross_border_flows_raw_data_df.empty:
        
# 3.1. ------------------------------------------------------------------------------ First file: just load it entirely
        cross_border_flows_raw_data_df = temp_df
    else:

# 3.1.1. ------------------------------ For subsequent files; Identify columns that already exist in the main DataFrame
        common_cols = temp_df.columns.intersection(cross_border_flows_raw_data_df.columns)
        
# 3.1.2. ------------------------------------------------- Identify which of these common columns have identical values
        cols_to_drop = []
        for col in common_cols:
            if temp_df[col].equals(cross_border_flows_raw_data_df[col]):
                cols_to_drop.append(col)
        
# 3.1.3. ---------------------------------------------- Drop the identical columns from the current file before joining
        temp_df_filtered = temp_df.drop(columns=cols_to_drop)
        
# 3.1.4. ---------------------------------------------------------------------------- Concatenate horizontally (axis=1)
        cross_border_flows_raw_data_df = pd.concat([cross_border_flows_raw_data_df, temp_df_filtered], axis=1)

# 4. ------------------------------------------------------------------------------------------------------------ Done 
print(f"Successfully loaded {len(files_to_read)} files for year {data_target_year}.")
print(f"Final DataFrame shape: {cross_border_flows_raw_data_df.shape}")
cross_border_flows_raw_data_df

Successfully loaded 2 files for year 2030.
Final DataFrame shape: (8760, 17)


,snapshot,AC_BE1 0-FR1 0,AC_BE1 0-NL1 0,AC_DE1 0-FR1 0,AC_DE1 0-NL1 0,DC_DE1 0-BE1 0,DC_GB0 0-NL1 0,DC_GB0 0-FR1 0,DC_GB0 0-GB2 0,DC_GB0 0-GB2 0.1,DC_GB0 0-FR1 0.1,DC_GB0 0-FR1 0.2,DC_FR1 0-GB0 0,DC_GB0 0-FR1 0.3,DC_GB0 0-DE1 0,DC_GB0 0-NL1 0.1,DC_GB0 0-BE1 0
0,2013-01-01 00:00:00,2466.246192,-6488.821910,1745.633719,-7796.836702,1349.994421,0.120328,2699.983739,0.130110,0.173650,1889.982974,2699.983610,0.013489,1889.982991,0.015597,0.122026,1349.990485
1,2013-01-01 01:00:00,2431.695357,-7061.422556,1711.865438,-8499.087372,1349.991073,131.163498,2699.983681,0.129975,0.173382,1889.982933,2699.983552,0.013515,1889.982950,0.103473,1278.981120,1349.990527
2,2013-01-01 02:00:00,2360.063276,-7023.054925,1656.391473,-8453.377258,1349.991066,84.486521,2699.983555,0.128852,0.172064,1889.982800,2699.983426,0.013599,1889.982817,0.134643,289.807757,1349.990431
3,2013-01-01 03:00:00,3158.836330,-6876.925450,2213.216320,-8283.573357,1349.990892,91.033439,2699.987461,0.124806,0.165472,1889.986850,2699.987331,0.010528,1889.986867,0.111490,365.999592,1349.990351
4,2013-01-01 04:00:00,3017.151325,-6749.901328,2116.505668,-8130.131626,1349.989318,1346.533201,2699.988175,0.130007,0.173317,1889.987551,2699.988045,0.009986,1889.987568,1889.093783,1616.656652,1349.990196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1923.743383,-5294.076648,1343.237531,-6363.520938,1349.978911,0.529509,2699.993928,0.040518,0.056921,1889.993501,2699.993798,0.005233,1889.993517,1889.992324,0.538342,1349.995054
8756,2013-12-31 20:00:00,1998.161420,-5355.262461,1403.261717,-6441.618818,1349.982151,0.533593,2699.993934,0.592574,0.835495,1889.993503,2699.993804,0.005233,1889.993519,1889.992259,0.543197,1349.995279
8757,2013-12-31 21:00:00,2136.934878,-5627.599642,1506.090982,-6758.496158,1349.984151,0.545051,2699.993927,0.736317,1.039180,1889.993498,2699.993797,0.005235,1889.993515,1889.991862,0.556427,1349.995281
8758,2013-12-31 22:00:00,2228.211407,-5519.676922,1567.272212,-6623.362281,1349.981305,0.504937,2699.993891,0.041862,0.058980,1889.993465,2699.993760,0.005262,1889.993481,1889.991761,0.513434,1349.995074


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
        6.2. Interconnected Zone Names Armonization
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Applying the DispaSET interconnection format; i.e. AA -> BB.
        <hr style="border: 0.5px solid skyblue;">
</div>

In [305]:
# 1. --------------------------------------------------- Create a reverse mapping for easy lookup: {Acronym: zone_name}
acronym_to_zone = {}
for zone in zone_names:
    acronyms = zone_names_equivalences_dict[zone].get('Acronym', [])
    
# 1.1. -------------------------------------------------------------------------------------- Ensure acronyms is a list
    if isinstance(acronyms, str): acronyms = [acronyms]
    
# 1.2. ---------------------------------------------------------------------------- Map the primary zone name to itself
    acronym_to_zone[zone] = zone

# 1.3. ------------------------------------------------------------------ Map all sub-acronyms to the primary zone name
    for acr in acronyms:
        acronym_to_zone[acr] = zone

# 2. ---------------------------- Sort by length (descending) to ensure longer acronyms are matched before shorter ones
all_patterns = sorted(acronym_to_zone.keys(), key=len, reverse=True)
pattern = "|".join(map(re.escape, all_patterns))

def process_header(header):
    
# 2.1. -------------------------------------------------------- Find all matches in the order they appear in the string
    matches = re.findall(pattern, header)
    
    if not matches:
        
# 2.2. -------------------------------------------------------- If no zone names or acronyms found, return header as is
        return header
    
# 2.3. ----------------- Map found acronyms to their primary zone names; Use a list to maintain the order of appearance
    cleaned_zones = []
    for m in matches:
        zone = acronym_to_zone[m]
        cleaned_zones.append(zone)
    
# 2.4. ----------------------------------------------------------------------------------------------- Join with ' -> '
    return " -> ".join(cleaned_zones)

# 3. Apply the transformation to the column names
cross_border_flows_raw_data_df.columns = [
    process_header(col) for col in cross_border_flows_raw_data_df.columns
]

# 4. ------------------------------------------------------------------------------------------------------------ Done 
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.")
cross_border_flows_raw_data_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.


,snapshot,BE -> FR,BE -> NL,DE -> FR,DE -> NL,DE -> BE,UK -> NL,UK -> FR,UK -> UK,UK -> UK,UK -> FR,UK -> FR,FR -> UK,UK -> FR,UK -> DE,UK -> NL,UK -> BE
0,2013-01-01 00:00:00,2466.246192,-6488.821910,1745.633719,-7796.836702,1349.994421,0.120328,2699.983739,0.130110,0.173650,1889.982974,2699.983610,0.013489,1889.982991,0.015597,0.122026,1349.990485
1,2013-01-01 01:00:00,2431.695357,-7061.422556,1711.865438,-8499.087372,1349.991073,131.163498,2699.983681,0.129975,0.173382,1889.982933,2699.983552,0.013515,1889.982950,0.103473,1278.981120,1349.990527
2,2013-01-01 02:00:00,2360.063276,-7023.054925,1656.391473,-8453.377258,1349.991066,84.486521,2699.983555,0.128852,0.172064,1889.982800,2699.983426,0.013599,1889.982817,0.134643,289.807757,1349.990431
3,2013-01-01 03:00:00,3158.836330,-6876.925450,2213.216320,-8283.573357,1349.990892,91.033439,2699.987461,0.124806,0.165472,1889.986850,2699.987331,0.010528,1889.986867,0.111490,365.999592,1349.990351
4,2013-01-01 04:00:00,3017.151325,-6749.901328,2116.505668,-8130.131626,1349.989318,1346.533201,2699.988175,0.130007,0.173317,1889.987551,2699.988045,0.009986,1889.987568,1889.093783,1616.656652,1349.990196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1923.743383,-5294.076648,1343.237531,-6363.520938,1349.978911,0.529509,2699.993928,0.040518,0.056921,1889.993501,2699.993798,0.005233,1889.993517,1889.992324,0.538342,1349.995054
8756,2013-12-31 20:00:00,1998.161420,-5355.262461,1403.261717,-6441.618818,1349.982151,0.533593,2699.993934,0.592574,0.835495,1889.993503,2699.993804,0.005233,1889.993519,1889.992259,0.543197,1349.995279
8757,2013-12-31 21:00:00,2136.934878,-5627.599642,1506.090982,-6758.496158,1349.984151,0.545051,2699.993927,0.736317,1.039180,1889.993498,2699.993797,0.005235,1889.993515,1889.991862,0.556427,1349.995281
8758,2013-12-31 22:00:00,2228.211407,-5519.676922,1567.272212,-6623.362281,1349.981305,0.504937,2699.993891,0.041862,0.058980,1889.993465,2699.993760,0.005262,1889.993481,1889.991761,0.513434,1349.995074


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Aggregating repeated columns.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [306]:
# 1. --------------------------------------------------------- Assuming cross_border_flows_raw_data_df is the DataFrame
df = cross_border_flows_raw_data_df.copy()

# 2. -------------------------------------------------------------------------------------- Find duplicate column names
duplicate_columns = df.columns[df.columns.duplicated(keep=False)].unique()

# 3. ------------------------------------------- For each duplicate column name, sum the values and create a new column
for col in duplicate_columns:

# 3.1. -------------------------------------------------------------------------- Get all columns with the current name
    cols_to_sum = df.loc[:, col].columns
    
# 3.2. ------------------------------------------------------------------------- Sum the values and create a new column
    df[col + '_sum'] = df[cols_to_sum].sum(axis=1)
    
# 3.3. -------------------------------------------------------------------------------------- Drop the original columns
    df = df.drop(columns=cols_to_sum)

# 4. ------------------------------------------------------------------- Rename the summed columns to the original name
df = df.rename(columns={col + '_sum': col for col in duplicate_columns})

# 5. ------------------------------------------------------------------------------------------- Result is stored in df
cross_border_flows_raw_data_df = df

# 6. ------------------------------------------------------------------------------------------------------------- Done 
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.")
cross_border_flows_raw_data_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.


,snapshot,BE -> FR,BE -> NL,DE -> FR,DE -> NL,DE -> BE,FR -> UK,UK -> DE,UK -> BE,UK -> NL,UK -> FR,UK -> UK
0,2013-01-01 00:00:00,2466.246192,-6488.821910,1745.633719,-7796.836702,1349.994421,0.013489,0.015597,1349.990485,0.484708,36719.733258,0.607520
1,2013-01-01 01:00:00,2431.695357,-7061.422556,1711.865438,-8499.087372,1349.991073,0.013515,0.103473,1349.990527,2820.289236,36719.732469,0.606714
2,2013-01-01 02:00:00,2360.063276,-7023.054925,1656.391473,-8453.377258,1349.991066,0.013599,0.134643,1349.990431,748.588557,36719.730390,0.601833
3,2013-01-01 03:00:00,3158.836330,-6876.925450,2213.216320,-8283.573357,1349.990892,0.010528,0.111490,1349.990351,914.066061,36719.794037,0.580557
4,2013-01-01 04:00:00,3017.151325,-6749.901328,2116.505668,-8130.131626,1349.989318,0.009986,1889.093783,1349.990196,5926.379705,36719.805355,0.606648
...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1923.743383,-5294.076648,1343.237531,-6363.520938,1349.978911,0.005233,1889.992324,1349.995054,2.135702,36719.898976,0.194876
8756,2013-12-31 20:00:00,1998.161420,-5355.262461,1403.261717,-6441.618818,1349.982151,0.005233,1889.992259,1349.995279,2.153580,36719.899040,2.856139
8757,2013-12-31 21:00:00,2136.934878,-5627.599642,1506.090982,-6758.496158,1349.984151,0.005235,1889.991862,1349.995281,2.202957,36719.898949,3.550993
8758,2013-12-31 22:00:00,2228.211407,-5519.676922,1567.272212,-6623.362281,1349.981305,0.005262,1889.991761,1349.995074,2.036742,36719.898388,0.201682


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Separating import and export flows.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [307]:
# 1. ---------------------------------------------------------------------- Make a copy to avoid modifying the original
df = cross_border_flows_raw_data_df.copy()

# 2. ----------------------------------------------------------------------- Identify flow columns vs. non-flow columns
flow_columns = []
non_flow_columns = []

for col in df.columns:
    if ' -> ' in col:
        parts = col.split(' -> ')
        if len(parts) == 2:
            source = parts[0].strip()
            target = parts[1].strip()
            
# 2.1. ------------------------------------------- Only consider it a flow column if both zones are valid and different
            if source in zone_names and target in zone_names and source != target:
                flow_columns.append(col)
                continue
                
# 2.2. ------------------------------------------------------------------ If not a valid flow column, treat as non-flow
    non_flow_columns.append(col)

# 3. ---------------------------------------------------------------------- build new derived columns from flow_columns
new_columns = {}  # maps column name -> list of Series (to allow duplicates)

# 4. ----------------------------------------------------------------------------------------- Process each flow column
for col in flow_columns:
    source, target = map(str.strip, col.split(' -> '))
    
# 4.1. ---------------------------------------------------------------------------------- Forward: positive values only
    forward_name = f"{source} -> {target}"
    forward_values = df[col].where(df[col] > 0)
    
# 4.2. ---------------------------------------------------------------------- Reverse: absolute of negative values only
    reverse_name = f"{target} -> {source}"
    reverse_values = (-df[col]).where(df[col] < 0)
    
# 4.3 --------------------------------------------------------- Append to lists (allowing duplicate column names later)
    new_columns[forward_name] = new_columns.get(forward_name, []) + [forward_values]
    new_columns[reverse_name] = new_columns.get(reverse_name, []) + [reverse_values]

# 5. ----------------------------------- Build final column list: first non-flow columns, then all derived flow columns
series_list = []
col_names = []

# 6. --------------------------------------------------------------------------------------- Add non-flow columns as-is
for col in non_flow_columns:
    series_list.append(df[col].reset_index(drop=True))
    col_names.append(col)

# 7. -------------------------------------------------------------- Add all derived flow columns (including duplicates)
for col_name, series_group in new_columns.items():
    for s in series_group:
        series_list.append(s.reset_index(drop=True))
        col_names.append(col_name)

# 8. ------------------------------------------------ Construct final DataFrame with intentional duplicate column names
final_array = np.column_stack([s.values for s in series_list])
cross_border_flows_raw_data_df = pd.DataFrame(final_array, columns=col_names, index=df.index)

# 9. ------------------------------------------------------------------------------------------------------------- Done 
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.")
cross_border_flows_raw_data_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.


,snapshot,UK -> UK,BE -> FR,FR -> BE,BE -> NL,NL -> BE,DE -> FR,FR -> DE,DE -> NL,NL -> DE,...,FR -> UK,FR -> UK,UK -> FR,UK -> FR,UK -> DE,DE -> UK,UK -> BE,BE -> UK,UK -> NL,NL -> UK
0,2013-01-01 00:00:00,0.60752,2466.246192,NaN,NaN,6488.82191,1745.633719,NaN,NaN,7796.836702,...,0.013489,NaN,NaN,36719.733258,0.015597,NaN,1349.990485,NaN,0.484708,NaN
1,2013-01-01 01:00:00,0.606714,2431.695357,NaN,NaN,7061.422556,1711.865438,NaN,NaN,8499.087372,...,0.013515,NaN,NaN,36719.732469,0.103473,NaN,1349.990527,NaN,2820.289236,NaN
2,2013-01-01 02:00:00,0.601833,2360.063276,NaN,NaN,7023.054925,1656.391473,NaN,NaN,8453.377258,...,0.013599,NaN,NaN,36719.73039,0.134643,NaN,1349.990431,NaN,748.588557,NaN
3,2013-01-01 03:00:00,0.580557,3158.83633,NaN,NaN,6876.92545,2213.21632,NaN,NaN,8283.573357,...,0.010528,NaN,NaN,36719.794037,0.11149,NaN,1349.990351,NaN,914.066061,NaN
4,2013-01-01 04:00:00,0.606648,3017.151325,NaN,NaN,6749.901328,2116.505668,NaN,NaN,8130.131626,...,0.009986,NaN,NaN,36719.805355,1889.093783,NaN,1349.990196,NaN,5926.379705,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,0.194876,1923.743383,NaN,NaN,5294.076648,1343.237531,NaN,NaN,6363.520938,...,0.005233,NaN,NaN,36719.898976,1889.992324,NaN,1349.995054,NaN,2.135702,NaN
8756,2013-12-31 20:00:00,2.856139,1998.16142,NaN,NaN,5355.262461,1403.261717,NaN,NaN,6441.618818,...,0.005233,NaN,NaN,36719.89904,1889.992259,NaN,1349.995279,NaN,2.15358,NaN
8757,2013-12-31 21:00:00,3.550993,2136.934878,NaN,NaN,5627.599642,1506.090982,NaN,NaN,6758.496158,...,0.005235,NaN,NaN,36719.898949,1889.991862,NaN,1349.995281,NaN,2.202957,NaN
8758,2013-12-31 22:00:00,0.201682,2228.211407,NaN,NaN,5519.676922,1567.272212,NaN,NaN,6623.362281,...,0.005262,NaN,NaN,36719.898388,1889.991761,NaN,1349.995074,NaN,2.036742,NaN


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Separating import and export flows.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [308]:
# 1. ------------------------------------------------------------------------------------- Make a copy of the DataFrame
df = cross_border_flows_raw_data_df.copy()

# 2. ----------------------------------------------------------------------------- REMOVE SELF-LOOP COLUMNS: "AA -> AA"
columns_to_drop = []
for col in df.columns:
    if ' -> ' in col:
        parts = col.split(' -> ')
        if len(parts) == 2:
            source, target = parts[0].strip(), parts[1].strip()
            if source == target and source in zone_names:
                columns_to_drop.append(col)

# 3. --------------------------------------------------------------------------------------- Drop all self-loop columns
df = df.drop(columns=columns_to_drop)

# 4. ------------------------------------------------------------------ Get all column names that appear more than once
duplicated_names = df.columns[df.columns.duplicated(keep=False)].unique()

# 5. ------------------------------------------------------------------ For each duplicated name, sum all its instances
for name in duplicated_names:
    
# 5.1. ------------------------------------------------------------------------------ Select all columns with this name
    subset = df.filter(regex=f'^{name}$')  # Exact match (avoids partial matches)
    # Sum across columns
    df[name + '_sum'] = subset.sum(axis=1)
    # Drop all original instances
    df = df.drop(columns=subset.columns)

# 6. ---------------------------------------------------------------------- RENAME summed columns back to original name
rename_map = {name + '_sum': name for name in duplicated_names}
df = df.rename(columns=rename_map)

# 7. ------------------------------------------------------------------------------------ Update the original DataFrame
cross_border_flows_raw_data_df = df

# 8. ------------------------------------------------------------------------------------------------------------- Done
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.")
cross_border_flows_raw_data_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.


,snapshot,BE -> FR,FR -> BE,BE -> NL,NL -> BE,DE -> FR,FR -> DE,DE -> NL,NL -> DE,DE -> BE,BE -> DE,UK -> DE,DE -> UK,UK -> BE,BE -> UK,UK -> NL,NL -> UK,FR -> UK,UK -> FR
0,2013-01-01 00:00:00,2466.246192,NaN,NaN,6488.82191,1745.633719,NaN,NaN,7796.836702,1349.994421,NaN,0.015597,NaN,1349.990485,NaN,0.484708,NaN,0.013489,36719.733258
1,2013-01-01 01:00:00,2431.695357,NaN,NaN,7061.422556,1711.865438,NaN,NaN,8499.087372,1349.991073,NaN,0.103473,NaN,1349.990527,NaN,2820.289236,NaN,0.013515,36719.732469
2,2013-01-01 02:00:00,2360.063276,NaN,NaN,7023.054925,1656.391473,NaN,NaN,8453.377258,1349.991066,NaN,0.134643,NaN,1349.990431,NaN,748.588557,NaN,0.013599,36719.73039
3,2013-01-01 03:00:00,3158.83633,NaN,NaN,6876.92545,2213.21632,NaN,NaN,8283.573357,1349.990892,NaN,0.11149,NaN,1349.990351,NaN,914.066061,NaN,0.010528,36719.794037
4,2013-01-01 04:00:00,3017.151325,NaN,NaN,6749.901328,2116.505668,NaN,NaN,8130.131626,1349.989318,NaN,1889.093783,NaN,1349.990196,NaN,5926.379705,NaN,0.009986,36719.805355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1923.743383,NaN,NaN,5294.076648,1343.237531,NaN,NaN,6363.520938,1349.978911,NaN,1889.992324,NaN,1349.995054,NaN,2.135702,NaN,0.005233,36719.898976
8756,2013-12-31 20:00:00,1998.16142,NaN,NaN,5355.262461,1403.261717,NaN,NaN,6441.618818,1349.982151,NaN,1889.992259,NaN,1349.995279,NaN,2.15358,NaN,0.005233,36719.89904
8757,2013-12-31 21:00:00,2136.934878,NaN,NaN,5627.599642,1506.090982,NaN,NaN,6758.496158,1349.984151,NaN,1889.991862,NaN,1349.995281,NaN,2.202957,NaN,0.005235,36719.898949
8758,2013-12-31 22:00:00,2228.211407,NaN,NaN,5519.676922,1567.272212,NaN,NaN,6623.362281,1349.981305,NaN,1889.991761,NaN,1349.995074,NaN,2.036742,NaN,0.005262,36719.898388


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Cross-border flow direction check
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [309]:
df = cross_border_flows_raw_data_df.copy()

# Step 1: Identify flow and non-flow columns
flow_cols = []
non_flow_cols = []

for col in df.columns:
    if ' -> ' in col:
        parts = col.split(' -> ')
        if len(parts) == 2:
            src = parts[0].strip()
            tgt = parts[1].strip()
            if src in zone_names and tgt in zone_names and src != tgt:
                flow_cols.append(col)
                continue
    non_flow_cols.append(col)

# CRITICAL: Ensure flow columns are numeric
for col in flow_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # Converts to float64, NaN for non-numeric

flow_col_set = set(flow_cols)
processed = set()

for col in flow_cols:
    src, tgt = map(str.strip, col.split(' -> '))
    mirror = f"{tgt} -> {src}"
    pair_key = tuple(sorted([src, tgt]))
    
    if mirror in flow_col_set and pair_key not in processed:
        processed.add(pair_key)
        
        # Now safe to fill NaN with 0 (columns are numeric)
        forward = df[col].fillna(0)
        reverse = df[mirror].fillna(0)
        
        net = forward - reverse
        df[col] = np.where(net >= 0, net, 0)
        df[mirror] = np.where(net < 0, -net, 0)

cross_border_flows_raw_data_df = df
print("✅ Net directional flows computed successfully.")
cross_border_flows_raw_data_df

✅ Net directional flows computed successfully.


,snapshot,BE -> FR,FR -> BE,BE -> NL,NL -> BE,DE -> FR,FR -> DE,DE -> NL,NL -> DE,DE -> BE,BE -> DE,UK -> DE,DE -> UK,UK -> BE,BE -> UK,UK -> NL,NL -> UK,FR -> UK,UK -> FR
0,2013-01-01 00:00:00,2466.246192,0.0,0.0,6488.821910,1745.633719,0.0,0.0,7796.836702,1349.994421,0.0,0.015597,0.0,1349.990485,0.0,0.484708,0.0,0.0,36719.719769
1,2013-01-01 01:00:00,2431.695357,0.0,0.0,7061.422556,1711.865438,0.0,0.0,8499.087372,1349.991073,0.0,0.103473,0.0,1349.990527,0.0,2820.289236,0.0,0.0,36719.718954
2,2013-01-01 02:00:00,2360.063276,0.0,0.0,7023.054925,1656.391473,0.0,0.0,8453.377258,1349.991066,0.0,0.134643,0.0,1349.990431,0.0,748.588557,0.0,0.0,36719.716791
3,2013-01-01 03:00:00,3158.836330,0.0,0.0,6876.925450,2213.216320,0.0,0.0,8283.573357,1349.990892,0.0,0.111490,0.0,1349.990351,0.0,914.066061,0.0,0.0,36719.783510
4,2013-01-01 04:00:00,3017.151325,0.0,0.0,6749.901328,2116.505668,0.0,0.0,8130.131626,1349.989318,0.0,1889.093783,0.0,1349.990196,0.0,5926.379705,0.0,0.0,36719.795369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1923.743383,0.0,0.0,5294.076648,1343.237531,0.0,0.0,6363.520938,1349.978911,0.0,1889.992324,0.0,1349.995054,0.0,2.135702,0.0,0.0,36719.893743
8756,2013-12-31 20:00:00,1998.161420,0.0,0.0,5355.262461,1403.261717,0.0,0.0,6441.618818,1349.982151,0.0,1889.992259,0.0,1349.995279,0.0,2.153580,0.0,0.0,36719.893807
8757,2013-12-31 21:00:00,2136.934878,0.0,0.0,5627.599642,1506.090982,0.0,0.0,6758.496158,1349.984151,0.0,1889.991862,0.0,1349.995281,0.0,2.202957,0.0,0.0,36719.893714
8758,2013-12-31 22:00:00,2228.211407,0.0,0.0,5519.676922,1567.272212,0.0,0.0,6623.362281,1349.981305,0.0,1889.991761,0.0,1349.995074,0.0,2.036742,0.0,0.0,36719.893125


In [297]:
cross_border_flows_raw_data_df.to_csv('/home/ray/Downloads/cross_border_flows_raw_data_df_0.csv', index=False)

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman;color:skyblue">
    2. Dispa-SET_Unleash Folder Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Determinning dynamically the zone_folder_path based on the location of the "Dispa-SET_Unleash" folder relative to the current working directory.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
If the "Dispa-SET_Unleash" folder is copied to a different machine or location, the dispaSET_unleash_folder_path variable will automatically adjust accordingly.
</div>

In [2]:
# Get the current working directory
current_directory = os.getcwd()

# Navigate to the parent directory of "Dispa-SET_Unleash"
dispaSET_unleash_parent_directory = os.path.dirname(current_directory)

# Get the path to the "Dispa-SET_Unleash" folder
dispaSET_unleash_folder_path = os.path.dirname(dispaSET_unleash_parent_directory)

# Construct the dispaSET_unleash_folder_name variable
dispaSET_unleash_folder_name = os.path.basename(dispaSET_unleash_folder_path)

print("dispaSET_unleash_folder_name:", dispaSET_unleash_folder_name)
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)

dispaSET_unleash_folder_name: Dispa-SET_Unleash
dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman;
    color:skyblue">
    3. Usefull Variable Definition
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Entering a value to all the variables which content are going to be used in some of the next stages of this script. 
</div>
<div style="text-align: left; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
Indicate the year of all data is referring to in the variable data_year.
</div>

In [3]:
# Year to which data refers to:
data_year = 2024

In [4]:
# Additional string to be appended
additional_path = "/RawData/CrossBorderFlows/"
additional_path_1 = "/RawData/CrossBorderFlows/Raw_Data_Sources/"

# Construct the Outage_Factors_folder_path variable
cross_border_flows_folder_path = dispaSET_unleash_folder_path + additional_path

# Construct the Outage_Factors_Raw_Data_folder_path variable
cross_border_flows_raw_data_folder_path = dispaSET_unleash_folder_path + additional_path_1

print("cross_border_flows_folder_path:", cross_border_flows_folder_path)
print("cross_border_flows_raw_data_folder_path:", cross_border_flows_raw_data_folder_path)

cross_border_flows_folder_path: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/
cross_border_flows_raw_data_folder_path: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/


<div style="background-color: black;">
<hr style="border: 1px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
    3.1. Back Up Directory
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Saving the original files into a Back up folder.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
Since in the next steps of the processing data new subfolders and files are going to be created, the original ones are saved in a back up foldet to return them as its default content ones the process will be finished.
</div>

In [5]:
additional_path_5 = '/RawData/CrossBorderFlows_backup/'

# Construct the backup_folder_path variable
backup_folder_path = dispaSET_unleash_folder_path + additional_path_5

print("backup_folder_path:", backup_folder_path)

# Create a backup of the directory
if os.path.exists(backup_folder_path):
    shutil.rmtree(backup_folder_path)  # Remove any existing backup if necessary
shutil.copytree(cross_border_flows_folder_path, backup_folder_path)

print(f"Backup created at {backup_folder_path}")

backup_folder_path: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows_backup/
Backup created at /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows_backup/


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman;color:skyblue">
    4. Country List Variable Definition
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Defining the list of countries according to the available data. 
</div>
<div style="text-align: jusitfy; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
All those countries that interchange flows with other countries different of the ones modelled in Dispa-SET are defined in the list.
</div>

In [6]:
# Define a list of country codes
cross_border_flows_per_unit_country_list = ["AL",
                                            "AT",
                                            "BA",
                                            "BE",
                                            "BG",
                                            "BY",
                                            "CH",
                                            "CY",
                                            "CZ",
                                            "DE",
                                            "DK",
                                            "EE",
                                            "ES",
                                            "FI",
                                            "FR",
                                            "GR",
                                            "HR",
                                            "HU",
                                            "IE",
                                            "IT",
                                            "LT",
                                            "LU",
                                            "LV",
                                            "MD",
                                            "ME",
                                            "MK",
                                            "MT",
                                            "NL",
                                            "NO",
                                            "PL",
                                            "PT",
                                            "RO",
                                            "RU",
                                            "RS",
                                            "SE",
                                            "SI",
                                            "SK",
                                            "TR",
                                            "UA",
                                            "UK"
                                           ]

In [7]:
# Define the directory and file path
file_name = 'country_list.csv'
file_path = os.path.join(cross_border_flows_raw_data_folder_path, file_name)

# Ensure the directory exists
os.makedirs(cross_border_flows_raw_data_folder_path, exist_ok=True)

# Create a DataFrame
df = pd.DataFrame(cross_border_flows_per_unit_country_list, columns=['Country_From'])

# Save the DataFrame to a CSV file
df.to_csv(file_path, index=False)

print(f"DataFrame saved to '{file_path}'")
cross_border_flows_country_list_file = file_path

DataFrame saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/country_list.csv'


<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Defining the list of countries modeled in Dispa-SET. 
</div>
<div style="text-align: jusitfy; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
Just a certain conutries in Dispa-SET are considered as those that exchange energy with the outside.
</div>

In [8]:
dispaSET_codes = ["AT", "BE", "BG", "CH", "CY", "CZ", "DE", "DK", "EE", "EL", "ES", "FI", "FR", "HR", "HU", 
                  "IE", "IT", "LT", "LU", "LV", "MT", "NL", "NO", "PL", "PT", "RO", "SE", "SI", "SK", "UK"
]

<div style="background-color: black;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Tracking Variables. 
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue"">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>

In [9]:
print (f"dispaSET_unleash_folder_name:                              {dispaSET_unleash_folder_name}")
print (f"dispaSET_unleash_folder_path:                              {dispaSET_unleash_folder_path}")
print (f"data_year:                                                 {data_year}")
print (f"cross_border_flows_folder_path:                            {cross_border_flows_folder_path}")   
print (f"cross_border_flows_raw_data_folder_path:                   {cross_border_flows_raw_data_folder_path}")
print (f"cross_border_flows_country_list_file:                      {cross_border_flows_country_list_file}")

dispaSET_unleash_folder_name:                              Dispa-SET_Unleash
dispaSET_unleash_folder_path:                              /home/ray/Dispa-SET_Unleash
data_year:                                                 2024
cross_border_flows_folder_path:                            /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/
cross_border_flows_raw_data_folder_path:                   /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/
cross_border_flows_country_list_file:                      /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/country_list.csv


<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 16px; font-family: TimesNewRoman;color:skyblue">
Defining the path to the sub-folders where all the cross border flows raw data is saved. 
</div>

In [10]:
def create_country_folder_column(cross_border_flows_country_list_file, cross_border_flows_raw_data_folder_path, data_year):
  """
  Creates a Country_Folder column in the specified CSV file.

  Args:
    cross_border_flows_country_list_file: The path to the CSV file.
    cross_border_flows_folder_path: The path to the cross-border flows folder.
    data_year: The data year (can be integer or string).
  """

  df = pd.read_csv(cross_border_flows_country_list_file)

  # Ensure data_year is a string before path joining
  data_year_str = str(data_year)

  def get_folder_path(country):
    return os.path.join(cross_border_flows_raw_data_folder_path, data_year_str, country)

  df['Country_Folder'] = df['Country_From'].apply(get_folder_path)
  df.to_csv(cross_border_flows_country_list_file, index=False)

  print("Country_Folder column created successfully.")

create_country_folder_column(cross_border_flows_country_list_file, cross_border_flows_raw_data_folder_path, data_year)

Country_Folder column created successfully.


<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Defining the neighbor countries. 
</div>

In [11]:
def create_neighbor_columns(cross_border_flows_country_list_file):
    """
    Creates neighbor columns based on CSV files in subfolders.

    Args:
        cross_border_flows_country_list_file: The path to the CSV file.
    """

    df = pd.read_csv(cross_border_flows_country_list_file)

    for index, row in df.iterrows():
        folder_path = row['Country_Folder']

        # Check if the folder exists, create it if not
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)

        csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
        csv_file_names = [f[:-4] for f in csv_files]

        for i, file_name in enumerate(csv_file_names):
            df.loc[index, f"Neighbor_{i+1}"] = file_name

        print(f"Processed folder: {folder_path}")

    df.to_csv(cross_border_flows_country_list_file, index=False)

create_neighbor_columns(cross_border_flows_country_list_file)

Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AL
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AT
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BA
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BE
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BG
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BY
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CH
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CY
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CZ
Processed folder: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/DE
Processed folder: /home/ray/Dispa-SET_Un

<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Since the Acronym of Grece in the downloaded data is 'GR' and the Dispa-SET format for the country is 'EL'. All the needed changes in the used variables are done.
<br>

In [12]:
# Read the CSV file into a DataFrame
df = pd.read_csv(cross_border_flows_country_list_file)

# Replace 'GR' with 'EL' in the entire DataFrame
df = df.applymap(lambda x: x.replace('GR', 'EL') if isinstance(x, str) else x)

# Save the updated DataFrame back to the CSV file
df.to_csv(cross_border_flows_country_list_file, index=False)

print(f"Replacements made and file saved: {cross_border_flows_country_list_file}")

Replacements made and file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/country_list.csv


/tmp/ipykernel_1144404/4086083268.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('GR', 'EL') if isinstance(x, str) else x)


In [13]:
def rename_files_and_directories(cross_border_flows_raw_data_folder_path, data_year):
    year_folder_path = os.path.join(cross_border_flows_raw_data_folder_path, str(data_year))

    for root, dirs, files in os.walk(year_folder_path, topdown=False):
        # Rename files
        for name in files:
            if 'GR' in name:
                new_name = name.replace('GR', 'EL')
                old_file_path = os.path.join(root, name)
                new_file_path = os.path.join(root, new_name)

                # Check if the new file already exists
                if not os.path.exists(new_file_path):
                    os.rename(old_file_path, new_file_path)
                    print(f"Renamed file: {old_file_path} to {new_file_path}")
                else:
                    print(f"File {new_file_path} already exists. Skipping renaming.")

        # Rename directories
        for name in dirs:
            if 'GR' in name:
                new_name = name.replace('GR', 'EL')
                old_dir_path = os.path.join(root, name)
                new_dir_path = os.path.join(root, new_name)

                # Check if the new directory already exists
                if not os.path.exists(new_dir_path):
                    os.rename(old_dir_path, new_dir_path)
                    print(f"Renamed directory: {old_dir_path} to {new_dir_path}")
                else:
                    print(f"Directory {new_dir_path} already exists. Skipping renaming.")

rename_files_and_directories(cross_border_flows_raw_data_folder_path, data_year)

Renamed file: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/MK/GR.csv to /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/MK/EL.csv
Renamed file: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AL/GR.csv to /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AL/EL.csv
Renamed file: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BG/GR.csv to /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BG/EL.csv
Renamed file: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/IT/GR.csv to /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/IT/EL.csv
Renamed file: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/TR/GR.csv to /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/TR/EL.csv
Renamed directory: /home/ray/Dispa-SET_Unleash/RawData/CrossBorde

<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
The downloaded files are joined into a single csv file under the name of the country which the flow comes from.
<br>

In [14]:
# Read the CSV file into a DataFrame
df = pd.read_csv(cross_border_flows_country_list_file)

# Ensure the column 'Country_Folder' exists
if 'Country_Folder' not in df.columns:
    raise ValueError("Column 'Country_Folder' does not exist in the CSV file")

# Function to join CSV files in a directory
def join_csv_files_in_directory(directory_path):
    csv_files = [f for f in os.listdir(directory_path) if f.endswith('.csv')]
    if not csv_files:
        return None
    
    # Read all CSV files into DataFrames
    dataframes = {csv_file: pd.read_csv(os.path.join(directory_path, csv_file)) for csv_file in csv_files}
    
    # Find the CSV file with the largest number of rows
    largest_file = max(dataframes, key=lambda x: len(dataframes[x]))
    base_df = dataframes[largest_file].iloc[:, :2].copy()
    base_df.columns = [base_df.columns[0], largest_file.replace('.csv', '')]
    
    # Merge the other CSV files based on the first column
    for csv_file, df in dataframes.items():
        if csv_file == largest_file:
            continue
        temp_df = df.iloc[:, [0, 1]]
        temp_df.columns = [temp_df.columns[0], csv_file.replace('.csv', '')]
        base_df = pd.merge(base_df, temp_df, on=base_df.columns[0], how='left')
    
    return base_df

# Create a new column for the paths of the new CSV files
df['Country_File_Path'] = ''

# Iterate through each row in the DataFrame
for index, row in df.iterrows():
    country_folder = row['Country_Folder']
    
    # Join CSV files in the directory
    joined_df = join_csv_files_in_directory(country_folder)
    
    if joined_df is not None:
        # Define the output file path
        output_file = os.path.join(country_folder, f"{os.path.basename(country_folder)}.csv")
        
        # Save the joined DataFrame to a new CSV file
        joined_df.to_csv(output_file, index=False)
        
        # Update the DataFrame with the path of the new CSV file
        df.at[index, 'Country_File_Path'] = output_file

        print(f"Joined CSV file saved to '{output_file}'")

# Save the updated DataFrame back to the main CSV file
df.to_csv(cross_border_flows_country_list_file, index=False)

print("All data has been processed and saved.")

Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AL/AL.csv'
Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AT/AT.csv'
Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BA/BA.csv'
Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BE/BE.csv'
Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BG/BG.csv'
Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BY/BY.csv'
Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CH/CH.csv'
Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CZ/CZ.csv'
Joined CSV file saved to '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/DE/

<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
The headers of these joined csv files are changed accordign the Dispa-SET cross border flow data format e.g. BE -> DE
</div>

In [15]:
# Read the main CSV file into a DataFrame
df = pd.read_csv(cross_border_flows_country_list_file)

# Ensure the required columns exist
if 'Country_From' not in df.columns or 'Country_File_Path' not in df.columns:
    raise ValueError("The CSV file must contain 'Country_From' and 'Country_File_Path' columns.")

# Function to update the headers of a CSV file
def update_csv_headers(file_path, new_header_prefix):
    # Read the CSV file into a DataFrame
    csv_df = pd.read_csv(file_path)
    
    # Get the current headers
    current_headers = csv_df.columns.tolist()
    
    # Create new headers for columns from the second column onward
    new_headers = [current_headers[0]] + [f"{new_header_prefix} -> {col}" for col in current_headers[1:]]
    
    # Update the DataFrame with the new headers
    csv_df.columns = new_headers
    
    # Save the updated DataFrame back to the CSV file
    csv_df.to_csv(file_path, index=False)
    print(f"Updated headers in '{file_path}'")

# Iterate through each row in the main DataFrame
for index, row in df.iterrows():
    country_from = row['Country_From']
    country_file_path = row['Country_File_Path']
    
    # Check if the file path is not empty and exists
    if pd.notna(country_file_path) and os.path.exists(country_file_path):
        update_csv_headers(country_file_path, country_from)
    else:
        print(f"File path '{country_file_path}' does not exist or is empty. Skipping...")

print("All CSV files have been processed.")

Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AL/AL.csv'
Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AT/AT.csv'
Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BA/BA.csv'
Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BE/BE.csv'
Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BG/BG.csv'
Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BY/BY.csv'
Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CH/CH.csv'
File path 'nan' does not exist or is empty. Skipping...
Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CZ/CZ.csv'
Updated headers in '/home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/D

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman;color:skyblue">
    6. Raw Data Format
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Addapting the time step data to the UTC for all the countries.
</div>

In [16]:
# Read the country list CSV file
country_list_df = pd.read_csv(cross_border_flows_country_list_file)

# Ensure the 'Country_File_Path' column exists
if 'Country_File_Path' not in country_list_df.columns:
    raise ValueError("Column 'Country_File_Path' does not exist in the CSV file")

# Define the function to convert time to UTC
def convert_to_utc(time_str):
    local_time = datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S%z')
    utc_time = local_time.astimezone(pytz.utc)
    return utc_time.strftime('%Y-%m-%d %H:%M:%S%z')

# Process each CSV file
for file_path in country_list_df['Country_File_Path'].dropna():
    # Ensure the file exists
    if not os.path.isfile(file_path):
        print(f"File not found: {file_path}")
        continue

    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Check if the 'index' column exists
    if 'index' not in df.columns:
        print(f"'index' column not found in file: {file_path}")
        continue

    # Convert the 'index' column to UTC
    df['index'] = df['index'].apply(convert_to_utc)
    
    # Save the updated CSV file
    df.to_csv(file_path, index=False)
    print(f"Updated file saved: {file_path}")

Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AL/AL.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AT/AT.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BA/BA.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BE/BE.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BG/BG.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BY/BY.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CH/CH.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CZ/CZ.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/DE/DE.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData

In [17]:
# Read the country list CSV file
country_list_df = pd.read_csv(cross_border_flows_country_list_file)

# Ensure the 'Country_File_Path' column exists
if 'Country_File_Path' not in country_list_df.columns:
    raise ValueError("Column 'Country_File_Path' does not exist in the CSV file")

# Function to update the year in the 'index' column
def update_index_year(df, data_year):
    # Ensure the 'index' column exists
    if 'index' not in df.columns:
        raise ValueError("'index' column not found in DataFrame")
    
    # Update the year in the 'index' column
    df['index'] = df['index'].apply(lambda x: f"{data_year}{x[4:]}" if str(x)[:4] != str(data_year) else x)
    
    return df

# Process each CSV file specified in the 'Country_File_Path' column
for file_path in country_list_df['Country_File_Path'].dropna():
    # Ensure the file exists
    if not os.path.isfile(file_path):
        print(f"File not found: {file_path}")
        continue
    
    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Ensure there are enough rows to move the first four rows to the last
    if len(df) < 4:
        print(f"Not enough rows to process in file: {file_path}")
        continue
    
    # Extract the first four rows (excluding headers)
    first_four_rows = df.iloc[:4].copy()
    
    # Drop the first four rows from the DataFrame
    df = df.iloc[4:].reset_index(drop=True)
    
    # Append the first_four_rows to the end of the DataFrame
    df = pd.concat([df, first_four_rows]).reset_index(drop=True)
    
    # Update the 'index' column year
    df = update_index_year(df, data_year)
    
    # Save the updated DataFrame back to the CSV file
    df.to_csv(file_path, index=False)
    print(f"Updated file saved: {file_path}")


Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AL/AL.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/AT/AT.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BA/BA.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BE/BE.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BG/BG.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/BY/BY.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CH/CH.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/CZ/CZ.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/Raw_Data_Sources/2024/DE/DE.csv
Updated file saved: /home/ray/Dispa-SET_Unleash/RawData

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman;color:skyblue">
    7. Cross Border Flows Clean File
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Joining all the cros border flows data to a single csv file with named as the analized year.
</div>

In [18]:
# Read the country list CSV file
country_list_df = pd.read_csv(cross_border_flows_country_list_file)

# Ensure the 'Country_File_Path' column exists
if 'Country_File_Path' not in country_list_df.columns:
    raise ValueError("Column 'Country_File_Path' does not exist in the CSV file")

# Process each CSV file specified in the 'Country_File_Path' column
file_paths = country_list_df['Country_File_Path'].dropna().tolist()

# Identify the CSV file with the largest number of rows
max_rows = 0
base_df = None
for file_path in file_paths:
    # Ensure the file exists
    if os.path.isfile(file_path):
        df = pd.read_csv(file_path)
        if len(df) > max_rows:
            max_rows = len(df)
            base_df = df.copy()

# If no base_df was found, raise an error
if base_df is None:
    raise ValueError("No valid CSV files found.")

# Initialize the combined DataFrame with the first column from the base DataFrame
combined_df = pd.DataFrame(base_df.iloc[:, 0])
combined_df.columns = [base_df.columns[0]]  # Keep the original name of the first column

# Add data from each CSV file to the combined DataFrame
for file_path in file_paths:
    if os.path.isfile(file_path):
        df = pd.read_csv(file_path)
        # Merge the data based on the first column
        combined_df = pd.merge(combined_df, df, on=base_df.columns[0], how='left')

# Save the combined DataFrame to a new CSV file named after the data_year variable
output_file_path = os.path.join(cross_border_flows_folder_path, f"{data_year}.csv")
combined_df.to_csv(output_file_path, index=False)
print(f"Combined CSV file saved: {output_file_path}")

Combined CSV file saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/2024.csv


<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Filling empty values.
</div>

In [19]:
def process_csv_file(file_path):
    """
    Processes a CSV file, replacing empty rows with '0'.

    Args:
        file_path: The path to the CSV file.
    """

    df = pd.read_csv(file_path)
    df.iloc[:, 1:] = df.iloc[:, 1:].fillna(0)
    df.to_csv(file_path, index=False)
    print(f"Processed file: {file_path}")

def process_folder(cross_border_flows_folder_path, data_year):
    """
    Processes CSV files within a folder.

    Args:
        cross_border_flows_folder_path: The path to the folder.
        data_year: The data year.
    """

    file_path = os.path.join(cross_border_flows_folder_path, f"{data_year}.csv")
    process_csv_file(file_path)

process_folder(cross_border_flows_folder_path, data_year)

Processed file: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/2024.csv


<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Dividing the clean data in time stepts of 15 minutes, 30 minutes, and 1 hour.
</div>

In [20]:
csv_file_path = os.path.join(cross_border_flows_folder_path, f'{data_year}.csv')

# Create the new directories
intervals = ['1h', '30min', '15min']
for interval in intervals:
    os.makedirs(os.path.join(cross_border_flows_folder_path, interval), exist_ok=True)

# Read the original CSV file
df = pd.read_csv(csv_file_path)

# Convert the 'index' column to datetime
df['index'] = pd.to_datetime(df['index'], format='%Y-%m-%d %H:%M:%S%z')

# Function to extract rows at a specific time step and save to a new CSV file
def extract_and_save(df, interval, folder_name):
    # Resample the DataFrame
    resampled_df = df.set_index('index').resample(interval).first().reset_index()
    
    # Define the new file path
    new_file_path = os.path.join(cross_border_flows_folder_path, folder_name, f'{data_year}.csv')
    
    # Save the resampled DataFrame to the new CSV file
    resampled_df.to_csv(new_file_path, index=False)
    print(f"File saved: {new_file_path}")

# Extract and save rows at different time steps
extract_and_save(df, '1H', '1h')
extract_and_save(df, '30T', '30min')
extract_and_save(df, '15T', '15min')

/tmp/ipykernel_1144404/2621242742.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  resampled_df = df.set_index('index').resample(interval).first().reset_index()


File saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/1h/2024.csv


/tmp/ipykernel_1144404/2621242742.py:17: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  resampled_df = df.set_index('index').resample(interval).first().reset_index()


File saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/30min/2024.csv


/tmp/ipykernel_1144404/2621242742.py:17: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  resampled_df = df.set_index('index').resample(interval).first().reset_index()


File saved: /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/15min/2024.csv


<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Aggregating all the exchanges between Dispaset Countries and the outside in two columns, i.e. XX -> RoW and RoW -> XX.
</div>

In [21]:
def process_cross_border_flows(dispaSET_codes, data_year, cross_border_flows_folder_path):
    # List of subfolders to check
    subfolders = ['1h', '30min', '15min']
    
    # Iterate over each subfolder
    for subfolder in subfolders:
        subfolder_path = os.path.join(cross_border_flows_folder_path, subfolder)
        csv_file_path = os.path.join(subfolder_path, f"{data_year}.csv")
        
        # Check if the subfolder and CSV file exist
        if os.path.exists(subfolder_path) and os.path.isfile(csv_file_path):
            print(f"Processing {csv_file_path}...")
            
            # Load CSV into a DataFrame
            df = pd.read_csv(csv_file_path)
            
            # Iterate over each code in dispaSET_codes to process headers
            for code in dispaSET_codes:
                # Identify columns for BE -> RoW and RoW -> BE
                to_row_cols = [col for col in df.columns if col.startswith(f"{code} ->") and not any(col.endswith(f"-> {c}") for c in dispaSET_codes)]
                from_row_cols = [col for col in df.columns if col.endswith(f"-> {code}") and not any(col.startswith(c + " ->") for c in dispaSET_codes)]
                
                # Sum and create the new RoW columns if there are any columns to aggregate
                if to_row_cols:
                    df[f"{code} -> RoW"] = df[to_row_cols].sum(axis=1)
                if from_row_cols:
                    df[f"RoW -> {code}"] = df[from_row_cols].sum(axis=1)
                
                # Drop the original columns after summing them
                df.drop(columns=to_row_cols + from_row_cols, inplace=True)
            
            # Keep only columns that have "RoW" in their name
            df = df[[col for col in df.columns if "RoW" in col or col == df.columns[0]]]
            
            # Save the modified DataFrame back to the CSV file
            df.to_csv(csv_file_path, index=False)
            print(f"Updated {csv_file_path}")
        else:
            print(f"Skipped {subfolder}: No CSV file found.")

# Run the function
process_cross_border_flows(dispaSET_codes, data_year, cross_border_flows_folder_path)

Processing /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/1h/2024.csv...
Updated /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/1h/2024.csv
Processing /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/30min/2024.csv...
Updated /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/30min/2024.csv
Processing /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/15min/2024.csv...
Updated /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/15min/2024.csv


<div style="background-color: black;">
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
Copying the time already formated Cross Border Flows data to the main Dispa-SET data base dirtectory
</div>

In [22]:
additional_path_2 = "/Database/CrossBorderFlows/"

# Construct the power_plants_raw_data_folder_path variable
cross_border_flows_data_base_folder_path = dispaSET_unleash_folder_path + additional_path_2

In [23]:
# Define the subfolder names
subfolders = ['1h', '30min', '15min']

# Function to copy files
def copy_files(data_year, source_base_path, dest_base_path, subfolders):
    for subfolder in subfolders:
        source_path = os.path.join(source_base_path, subfolder, f"{data_year}.csv")
        dest_folder_path = os.path.join(dest_base_path, subfolder)

        # Create the destination subfolder if it does not exist
        os.makedirs(dest_folder_path, exist_ok=True)

        dest_path = os.path.join(dest_folder_path, f"{data_year}.csv")
        
        # Copy the file
        if os.path.isfile(source_path):
            shutil.copy2(source_path, dest_path)
            print(f"Copied {source_path} to {dest_path}")
        else:
            print(f"File {source_path} does not exist")

# Call the function
copy_files(data_year, cross_border_flows_folder_path, cross_border_flows_data_base_folder_path, subfolders)

Copied /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/1h/2024.csv to /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/1h/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/30min/2024.csv to /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/30min/2024.csv
Copied /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows/15min/2024.csv to /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/15min/2024.csv


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
8. Cross Border Flows Folder Back Up
</div>
<div style="text-align: justify; margin-left: 0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Once all the formating process was done the Cross Border Flows Folder is restored to its defoult state.
</div>

In [24]:
if os.path.exists(cross_border_flows_folder_path):
    shutil.rmtree(cross_border_flows_folder_path)  # Remove the current directory
shutil.copytree(backup_folder_path, cross_border_flows_folder_path)

print(f"Directory restored to original state from {backup_folder_path}")

Directory restored to original state from /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows_backup/


In [25]:
shutil.rmtree(backup_folder_path)
print(f"Backup folder {backup_folder_path} deleted successfully.")

Backup folder /home/ray/Dispa-SET_Unleash/RawData/CrossBorderFlows_backup/ deleted successfully.
